In [9]:
# Transilien BI Project

## Objective
"""
Analyze train punctuality and identify influencing factors such as:
- weather
- strikes
- holidays
- temporal patterns
"""

## Pipeline
"""
Raw data → Cleaning → Enrichment → Export for SAP Analytics Cloud
"""

'\nRaw data → Cleaning → Enrichment → Export for SAP Analytics Cloud\n'

In [3]:
## Import libraries
"""
We import the libraries required for:
- data manipulation
- feature engineering
- holiday generation
"""
import pandas as pd
import holidays
import matplotlib.pyplot as plt
import numpy as np

In [4]:
## Load the Transilien dataset
"""
The original dataset contains monthly punctuality indicators for Transilien lines.
"""
df = pd.read_csv("../Data/processed/ponctualite-mensuelle-transilien.csv",sep=";")
df.head()

,Date,Service,Ligne,Nom de la ligne,Taux de ponctualité,Nombre de voyageurs à l'heure pour un voyageur en retard
0,2013-01,RER,A,RER A,83.6,5.1
1,2013-01,Transilien,R,Paris Sud Est,87.2,6.8
2,2013-03,Transilien,H,Paris Nord Ouest,92.3,12.0
3,2013-04,Transilien,N,Paris Montparnasse,90.2,9.2
4,2013-05,RER,D,RER D,87.1,6.8


In [7]:
## Data cleaning
"""
We standardize column names and convert data types to prepare the dataset for analysis.
"""
df.columns =(
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("'","_")
    .str.replace("é","e")

    )
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 6 columns):
 #   Column                                                    Non-Null Count  Dtype  
---  ------                                                    --------------  -----  
 0   date                                                      2009 non-null   str    
 1   service                                                   2009 non-null   str    
 2   ligne                                                     2009 non-null   str    
 3   nom_de_la_ligne                                           2009 non-null   str    
 4   taux_de_ponctualite                                       2008 non-null   float64
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  2009 non-null   float64
dtypes: float64(2), str(4)
memory usage: 94.3 KB


In [ ]:
""" à ajouter en anglais"""
df["date"]=pd.to_datetime(df["date"])
df = df.dropna(subset="date")
df.info()



<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 6 columns):
 #   Column                                                    Non-Null Count  Dtype         
---  ------                                                    --------------  -----         
 0   date                                                      2009 non-null   datetime64[us]
 1   service                                                   2009 non-null   str           
 2   ligne                                                     2009 non-null   str           
 3   nom_de_la_ligne                                           2009 non-null   str           
 4   taux_de_ponctualite                                       2008 non-null   float64       
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  2009 non-null   float64       
dtypes: datetime64[us](1), float64(2), str(3)
memory usage: 94.3 KB


,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,False
3,False,False,False,False,False,False
4,False,False,False,False,False,False
...,...,...,...,...,...,...
2004,False,False,False,False,False,False
2005,False,False,False,False,False,False
2006,False,False,False,False,False,False
2007,False,False,False,False,False,False


In [18]:
## Feature engineering
"""
We create temporal variables and KPIs to improve BI analysis.
"""
df["annee"]=df["date"].dt.year
df["mois"]=df["date"].dt.month
df["nom_mois"]=df["date"].dt.month_name()
df["trimestre"]=df["date"].dt.quarter
#df["jour"]=df["jour"].dt.day_name()
df["taux_irregularite"] = 100 - df["taux_de_ponctualite"]

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 11 columns):
 #   Column                                                    Non-Null Count  Dtype         
---  ------                                                    --------------  -----         
 0   date                                                      2009 non-null   datetime64[us]
 1   service                                                   2009 non-null   str           
 2   ligne                                                     2009 non-null   str           
 3   nom_de_la_ligne                                           2009 non-null   str           
 4   taux_de_ponctualite                                       2008 non-null   float64       
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  2009 non-null   float64       
 6   annee                                                     2009 non-null   int32         
 7   mois                                                 

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite
0,2013-01-01,RER,A,RER A,83.6,5.1,2013,1,January,1,16.4
1,2013-01-01,Transilien,R,Paris Sud Est,87.2,6.8,2013,1,January,1,12.8
2,2013-03-01,Transilien,H,Paris Nord Ouest,92.3,12.0,2013,3,March,1,7.7
3,2013-04-01,Transilien,N,Paris Montparnasse,90.2,9.2,2013,4,April,2,9.8
4,2013-05-01,RER,D,RER D,87.1,6.8,2013,5,May,2,12.9


In [ ]:
## Holiday enrichment
"""
French holidays are generated dynamically using the holidays Python package.
"""
